In [1]:
GPT_CONFIG = {
    "vocab_size": 50257,
    "num_layers": 12,
    "num_heads": 12,
    "emb_dim": 768,
    "context_length": 1024,
    "dropout": 0.1,
    "num_classes": 2,
    'bias':False
}

In [2]:
import tiktoken
import torch
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch.shape)

torch.Size([2, 4])


In [3]:
import torch.nn as nn
torch.manual_seed(123)
batch_sample = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_sample)
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
out_norm = (out - mean) / torch.sqrt(var)
print(out_norm)
torch.set_printoptions(precision=4, sci_mode=False)
print(out_norm.mean(dim=-1, keepdim=True))
print(out_norm.var(dim=-1, keepdim=True))

tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
tensor([[    0.0000],
        [    0.0000]], grad_fn=<MeanBackward1>)
tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [8]:
class LayerNorm(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.eps = 1e-5
        self.gamma = nn.Parameter(torch.ones(d_in))
        self.beta = nn.Parameter(torch.zeros(d_in))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / (torch.sqrt(var) + self.eps)
        return self.gamma * norm_x + self.beta

In [15]:
ln = LayerNorm(d_in=5)
out_ln = ln(batch_sample)
mean = out_ln.mean(dim=-1)
var = out_ln.var(dim=-1, unbiased=False)
print(mean, var)

tensor([    -0.0000,      0.0000], grad_fn=<MeanBackward1>) tensor([1.0000, 1.0000], grad_fn=<VarBackward0>)


In [7]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim']),
            GELU(),
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim']),
        )

    def forward(self, x):
        return self.layers(x)

ffn = FeedForward(GPT_CONFIG)
x = torch.rand(2, 3, GPT_CONFIG['emb_dim'])
out = ffn(x)
print(out)

tensor([[[ 0.0807,  0.0853,  0.0298,  ...,  0.1046, -0.3081, -0.1246],
         [ 0.0604,  0.0350,  0.0455,  ...,  0.0865, -0.1525, -0.0411],
         [ 0.0810,  0.0246,  0.0091,  ...,  0.0999, -0.1398, -0.0793]],

        [[ 0.0142,  0.0708,  0.1073,  ...,  0.0364, -0.1319, -0.1204],
         [ 0.1021,  0.0433,  0.0930,  ...,  0.0534, -0.2062, -0.1041],
         [ 0.0952,  0.0192,  0.0695,  ...,  0.0938, -0.2119, -0.1244]]],
       grad_fn=<ViewBackward0>)


In [58]:
class ExampleDeepNN(nn.Module):
    def __init__(self, layer_sizes, use_shortcut=False):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])
    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            if self.use_shortcut and x.shape == out.shape:
                x = x + out
            else:
                x = out
        return x

layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([1., 0., -1.])
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNN(layer_sizes, use_shortcut=False)
model_with_shortcut = ExampleDeepNN(layer_sizes, use_shortcut=True)
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])
    loss = nn.MSELoss()
    loss = loss(output, target)
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"Gradients for {name}: {param.grad.abs().mean().item()}")
# print_gradients(model_with_shortcut, sample_input)

In [59]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNN(layer_sizes, use_shortcut=True)
print_gradients(model_with_shortcut, sample_input)

Gradients for layers.0.0.weight: 0.22169791162014008
Gradients for layers.1.0.weight: 0.20694102346897125
Gradients for layers.2.0.weight: 0.32896995544433594
Gradients for layers.3.0.weight: 0.2665732204914093
Gradients for layers.4.0.weight: 1.3258541822433472


In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(d_in=cfg['emb_dim'], d_out=cfg['emb_dim'], context_length=cfg['context_length'], 
                                            dropout=cfg['dropout'], num_heads=cfg['num_heads'], bias=cfg['bias'])
        self.ffn = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.dropout = nn.Dropout(cfg['dropout'])
    
    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        print(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x = x + shortcut
        return x

torch.manual_seed(123)
x = torch.rand(2, 4, GPT_CONFIG['emb_dim'])
block = TransformerBlock(GPT_CONFIG)
out = block(x)
# print(out)

tensor([[[ 0.0475,  0.0953, -0.2161,  ...,  0.2032, -0.2954,  0.1886],
         [-0.0976, -0.0897, -0.0440,  ..., -0.0691, -0.4102,  0.1129],
         [-0.1272,  0.1372, -0.1857,  ...,  0.0341, -0.2393,  0.1237],
         [-0.0203, -0.0684, -0.1266,  ..., -0.3517, -0.1095,  0.1305]],

        [[-0.5220,  0.2147, -0.0446,  ..., -0.0373, -0.1554, -0.6017],
         [-0.2275,  0.2052,  0.0740,  ..., -0.0152,  0.2024, -0.3165],
         [-0.0618, -0.0686,  0.0686,  ..., -0.2016,  0.0296, -0.4385],
         [ 0.1656, -0.1838, -0.0162,  ..., -0.1162, -0.0292,  0.0172]]],
       grad_fn=<ViewBackward0>)


## 多头注意力+层归一化+前馈神经网络+残差链接 = Tranformer

In [10]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_embedding = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_embedding = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.dropout = nn.Dropout(cfg['dropout'])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg['num_layers'])] # 4
        )
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_embedding(in_idx)
        pos_embeds = self.pos_embedding(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.dropout(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [11]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG)
out = model(batch)
print(out)


tensor([[[-0.3091, -0.4685,  0.1402,  ..., -0.1073,  0.2294,  0.2545],
         [-0.8277, -0.2274,  0.0125,  ...,  0.0332,  0.2589,  0.3776],
         [-0.3239, -0.2419,  0.0488,  ..., -0.2091,  0.3337,  0.0841],
         [-0.1742, -0.3053, -0.0806,  ..., -0.1049, -0.0011,  0.0666]],

        [[-0.0539, -0.3594,  0.1552,  ...,  0.1561, -0.2729,  0.3169],
         [-0.2146, -0.0297,  0.1153,  ...,  0.1914, -0.2186,  0.4676],
         [-0.1682, -0.1659,  0.2510,  ...,  0.2738,  0.1684,  0.3408],
         [-0.0448,  0.0125,  0.0402,  ...,  0.3494, -0.0032,  0.0550]]],
       grad_fn=<ViewBackward0>)
tensor([[[ 0.2471,  0.1274,  0.0866,  ..., -0.0651,  0.0014, -0.1364],
         [ 0.0747,  0.0547,  0.0531,  ..., -0.2368,  0.2500, -0.0394],
         [ 0.1754,  0.2755, -0.1467,  ..., -0.0363,  0.3751, -0.0175],
         [ 0.0290,  0.2732,  0.1085,  ...,  0.1376,  0.2722, -0.1899]],

        [[ 0.0082,  0.5120, -0.0589,  ...,  0.0825,  0.2270, -0.0651],
         [-0.0477,  0.0221,  0.4860,  .

In [77]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 163,050,577


In [84]:
ffn = FeedForward(GPT_CONFIG)
print(f"FeedForward parameters: {sum(p.numel() for p in ffn.parameters() if p.requires_grad):,}")
multi_attn = MultiHeadAttention(d_in=GPT_CONFIG['emb_dim'], d_out=GPT_CONFIG['emb_dim'], context_length=GPT_CONFIG['context_length'], dropout=GPT_CONFIG['dropout'], num_heads=GPT_CONFIG['num_heads'])
print(f"MultiHeadAttention parameters: {sum(p.numel() for p in multi_attn.parameters() if p.requires_grad):,}")

FeedForward parameters: 4,722,432
MultiHeadAttention parameters: 2,359,296


In [12]:
def generate_text(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
        
    return idx

In [13]:
torch.manual_seed(123)
context = "Hello, I am"
encoded = tokenizer.encode(context)
encode_tensor = torch.tensor(encoded).unsqueeze(0)
model.eval()
out = generate_text(
    model=model,
    idx=encode_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG['context_length']
)
print(encode_tensor)
print(out)
decode_text = tokenizer.decode(out.squeeze(0).tolist())
print(decode_text)

tensor([[[-0.8231, -0.0712,  0.1163,  ..., -0.3008,  0.0665,  0.5461],
         [-0.4259, -0.1714,  0.3074,  ..., -0.0644,  0.0853,  0.3472],
         [-0.3766, -0.4054,  0.3280,  ..., -0.2183,  0.3885,  0.2661],
         [-0.3000, -0.2909,  0.0634,  ..., -0.0518,  0.0813,  0.1879]]])
tensor([[[ 0.3969,  0.1280, -0.4097,  ..., -0.0303,  0.4214,  0.0588],
         [ 0.2074,  0.1958, -0.1087,  ..., -0.3332,  0.2825,  0.0343],
         [ 0.1367,  0.2547, -0.0950,  ..., -0.0818,  0.2490, -0.0588],
         [ 0.1617,  0.1993,  0.0580,  ..., -0.1455,  0.0551,  0.0216]]])
tensor([[[-0.0532,  0.0739, -0.3869,  ...,  0.1890, -0.2707,  0.4490],
         [-0.1852, -0.0571, -0.4853,  ...,  0.2067,  0.0944, -0.0734],
         [-0.3321, -0.1419, -0.2013,  ...,  0.1394,  0.3130,  0.0592],
         [-0.3077, -0.0757,  0.0099,  ...,  0.1973,  0.1232, -0.0454]]])
tensor([[[ 0.2879, -0.3458, -0.2393,  ..., -0.1860,  0.3873, -0.0533],
         [ 0.5031, -0.1107, -0.1236,  ..., -0.0232,  0.1513, -0.0692],
